In [7]:
! pip install brian2 -q

In [8]:
# Keep numerical outputs and figures inside 01_Original_Colab/results.
from pathlib import Path

ROOT_CANDIDATES = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    (p for p in ROOT_CANDIDATES if (p / "01_Original_Colab").is_dir()),
    next((p.parent for p in ROOT_CANDIDATES if p.name == "01_Original_Colab"), Path.cwd()),
)
RESULTS_ROOT = ROOT / "01_Original_Colab" / "results"
CONDITION = "G4.0"
OUTPUT_DIR = RESULTS_ROOT / CONDITION
FIGURES_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


In [9]:
from brian2 import *
import numpy as np
import matplotlib.pyplot as plt

from scipy.signal import welch
from scipy.integrate import quad
from scipy.special import erfc
from scipy.optimize import brentq
from scipy.fft import fft
from scipy.ndimage import gaussian_filter1d

import warnings

warnings.filterwarnings('ignore')
prefs.codegen.target = 'cython'

In [10]:
start_scope()
defaultclock.dt = 0.1 * ms           # matches the paper's Euler step (Delta t = 0.1 ms)
seed(4321)                           # global seed: fixes connectivity AND initial voltages

# --- Population sizes -------------------------------------------------------
N_E      = 100000                    # excitatory neurons  -> C_E = 1000
gamma    = 0.25
N_I      = int(gamma * N_E)          # inhibitory neurons  = 2500
N        = N_E + N_I                 # total               = 12500

# --- Sparse connectivity ---------------------------------------------------
epsilon  = 0.01                       # connection probability
C_E      = int(epsilon * N_E)        # exact # excitatory presyn.  = 1000
C_I      = int(epsilon * N_I)        # exact # inhibitory presyn.  = 250

# --- Neuron parameters ------------------------------------------------------
tau     = 20 * ms                    # membrane time constant
theta   = 20 * mV                    # firing threshold
V_r     = 10  * mV                    # reset potential
tau_rp  = 2  * ms                    # absolute refractory period

# --- Synapse parameters -----------------------------------------------------
J   = 1.0 * mV                       # excitatory PSP amplitude
g   = 4.0                            # relative inhibitory strength
D   = 1.5 * ms                       # synaptic transmission delay
RI  = 30 * mV                        # Constant external input

# --- Neuron Group Setup -----------------------------------------------------
neurons = NeuronGroup(
    N,
    model="dv/dt = (-v + RI) / tau : volt (unless refractory)",
    threshold="v > theta",
    reset="v = V_r",
    refractory=tau_rp,
    method="euler",
)
neurons.v = "rand() * mV"            # random initial voltages in [0, 1] mV

exc_neurons = neurons[:N_E]
inh_neurons = neurons[N_E:]

In [11]:
# --- Recurrent synapses (NumPy Fixed In-Degree) -----------------------------
exc_syn = Synapses(exc_neurons, neurons, on_pre="v += J", delay=D)
inh_syn = Synapses(inh_neurons, neurons, on_pre="v += -g*J", delay=D)

# 1. Pre-calculate targets: Every neuron in N gets exactly C_E (or C_I) inputs
targets_E = np.repeat(np.arange(N), C_E)
targets_I = np.repeat(np.arange(N), C_I)

# 2. Pre-calculate sources: Randomly sample from presynaptic pools
# np.random.randint is exceptionally fast and standard for sparse network practice
sources_E = np.random.randint(0, N_E, size=N * C_E)
sources_I = np.random.randint(0, N_I, size=N * C_I)

# 3. Connect synapses directly using the generated NumPy arrays
exc_syn.connect(i=sources_E, j=targets_E)
inh_syn.connect(i=sources_I, j=targets_I)


In [12]:
# --- Monitors & Execution ---------------------------------------------------
sim_time = 3000 * ms
rate_mon   = PopulationRateMonitor(neurons)
spike_mon  = SpikeMonitor(neurons)

run(sim_time)

In [13]:
# Extract spike trains from the monitor
spike_trains = spike_mon.spike_trains()
cv_list = []

# Iterate through each neuron's spike train
for neuron_idx, train in spike_trains.items():

    # Ensure there are enough spikes to calculate an ISI
    if len(train) > 2:

        # Calculate Inter-Spike Intervals (ISIs)
        isi = np.diff(train / ms)

        # Calculate CV: Standard Deviation / Mean
        cv = np.std(isi) / np.mean(isi)
        cv_list.append(cv)

# Calculate the network-wide average CV
if cv_list:
    mean_cv = np.mean(cv_list)
    print(f"Average Network CV: {mean_cv:.3f}")
else:
    print("Not enough spikes to calculate CV.")


average_firing_rate = np.mean(rate_mon.rate)
print(f"Average Firing Rate: {average_firing_rate:.2f} Hz")

Average Network CV: 3.764
Average Firing Rate: 222.57 Hz
